## 🎯 Learning Objectives
* Understand the fundamental concepts of distributed training for large language models (LLMs).
* Differentiate between data parallelism and model parallelism, including their mechanisms and trade-offs.
* Grasp the conceptual implementation of data parallelism using PyTorch's distributed backend.
* Identify scenarios where data parallelism, model parallelism, or hybrid approaches are most appropriate.
* Become familiar with modern distributed training frameworks and their role in LLM development.


## Distributed Training Fundamentals: Data Parallelism and Model Parallelism

Training state-of-the-art Large Language Models (LLMs) like GPT-4, Llama 3, or Gemini requires an immense amount of computational power and memory. A single GPU, even the most powerful ones available in 2026, simply cannot hold the entire model parameters and intermediate activations for models with hundreds of billions or even trillions of parameters, nor can it process the vast datasets efficiently within a reasonable timeframe. This is where **distributed training** becomes indispensable.

Distributed training involves spreading the computational workload across multiple devices (GPUs) and/or multiple machines (nodes), allowing us to train models that are too large for a single device or to accelerate training significantly. The two primary paradigms for achieving this are **Data Parallelism** and **Model Parallelism**.

### 1. Data Parallelism

Imagine you have a complex recipe (your LLM) and a massive pile of ingredients (your training data). Instead of one chef trying to cook everything alone, you hire multiple chefs. Each chef gets a copy of the *entire recipe* (the full model) but is given only a *portion of the ingredients* (a shard of the data). Each chef cooks their portion independently, calculates what adjustments are needed (gradients), and then all chefs come together to agree on the final adjustments before starting the next batch. This is the essence of data parallelism.

**Mechanism:**
1.  **Model Replication:** The full model is replicated on each participating device (GPU).
2.  **Data Sharding:** The training dataset is divided into smaller, non-overlapping batches. Each device receives a unique mini-batch.
3.  **Local Forward/Backward Pass:** Each device performs a forward pass and a backward pass on its local mini-batch, computing gradients for its replica of the model.
4.  **Gradient Aggregation (All-Reduce):** After computing local gradients, all devices communicate to average or sum their gradients. This is typically done using an `all-reduce` operation, ensuring that all model replicas end up with identical, synchronized gradients.
5.  **Parameter Update:** Each device then updates its local model replica's parameters using the aggregated gradients.

**Key Characteristics:**
*   **Scales Computation:** Effectively increases the batch size, leading to faster training throughput.
*   **Memory:** Each device must be able to fit the *entire model* in its memory.
*   **Communication:** Primarily involves communicating gradients (which can be large) between devices.

**Modern Implementations:** PyTorch's `DistributedDataParallel` (DDP) and especially **Fully Sharded Data Parallel (FSDP)** are popular choices. FSDP takes data parallelism a step further by sharding not just the data, but also the model's parameters, gradients, and optimizer states across devices, significantly reducing memory footprint per GPU and enabling training of much larger models than DDP.

### 2. Model Parallelism

Now, imagine you're building a very complex product on an assembly line. Instead of each worker building the entire product, each worker specializes in one specific step or component. Worker A attaches the base, passes it to Worker B, who adds the engine, passes it to Worker C, who installs the electronics, and so on. The product (data/activations) moves sequentially through the workers (model layers/parts).

**Mechanism:**
1.  **Model Partitioning:** The model's layers or components are split across multiple devices. For example, layers 0-10 might be on GPU 0, layers 11-20 on GPU 1, etc.
2.  **Sequential Execution:** Input data is fed to the first device, which processes its part of the model and passes the intermediate activations to the next device. This continues until the final output is produced.
3.  **Backward Pass:** Gradients flow backward through the same partitioned model, with each device computing gradients for its assigned layers.

**Key Characteristics:**
*   **Scales Memory:** Allows training models that are too large to fit on a single device.
*   **Communication:** Primarily involves communicating intermediate activations (forward pass) and gradients (backward pass) between devices.
*   **Load Balancing:** Requires careful partitioning to ensure each device has a roughly equal workload to avoid 


pipeline bubbles


 (idle time).

**Types of Model Parallelism:**
*   **Pipeline Parallelism:** Divides the model *sequentially* by layers. Different mini-batches can be processed in a pipeline fashion to reduce idle time.
*   **Tensor Parallelism (or Intra-layer Parallelism):** Divides individual layers (e.g., large linear layers or attention mechanisms) across multiple devices. For example, a large weight matrix `W` might be split into `W1` and `W2`, with each part processed on a different device, and their results combined.

**Modern Implementations:** Frameworks like **DeepSpeed** (with ZeRO stages 2 and 3, and pipeline parallelism), **Megatron-LM**, and PyTorch's native **`torch.distributed.pipeline`** and **`torch.distributed.tensor.parallel`** modules are used for model parallelism.

### Hybrid Parallelism

For truly massive LLMs, a combination of both data and model parallelism is often employed. For instance, you might use model parallelism to split a very large model across a few nodes, and then use data parallelism within each node (across its GPUs) to process different data shards. This allows for both memory scaling and throughput scaling simultaneously.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.distributed as dist
import torch.multiprocessing as mp
import os

# --- Configuration --- #
WORLD_SIZE = 2 # Simulate 2 GPUs/processes
BATCH_SIZE_PER_RANK = 4
GLOBAL_BATCH_SIZE = BATCH_SIZE_PER_RANK * WORLD_SIZE
INPUT_DIM = 128
HIDDEN_DIM = 256
OUTPUT_DIM = 10

# --- 1. Data Parallelism (Conceptual Simulation) --- #

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(INPUT_DIM, HIDDEN_DIM)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(HIDDEN_DIM, OUTPUT_DIM)

    def forward(self, x):
        return self.layer2(self.relu(self.layer1(x)))

def run_data_parallel_rank(rank, world_size):
    # Initialize distributed environment
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '29500'
    dist.init_process_group("nccl", rank=rank, world_size=world_size)

    print(f"Rank {rank} initialized for data parallelism.")

    model = SimpleModel().to(rank) # Model replica on each GPU
    optimizer = optim.SGD(model.parameters(), lr=0.01)
    criterion = nn.CrossEntropyLoss()

    # Simulate sharded data
    # In a real scenario, a DistributedSampler would handle this
    dummy_data = torch.randn(GLOBAL_BATCH_SIZE, INPUT_DIM)
    dummy_labels = torch.randint(0, OUTPUT_DIM, (GLOBAL_BATCH_SIZE,))

    # Each rank gets a slice of the global batch
    start_idx = rank * BATCH_SIZE_PER_RANK
    end_idx = (rank + 1) * BATCH_SIZE_PER_RANK
    local_data = dummy_data[start_idx:end_idx].to(rank)
    local_labels = dummy_labels[start_idx:end_idx].to(rank)

    # Forward pass
    outputs = model(local_data)
    loss = criterion(outputs, local_labels)

    # Backward pass (computes local gradients)
    loss.backward()

    # All-reduce gradients
    # This aggregates gradients from all ranks
    for param in model.parameters():
        if param.grad is not None:
            # All-reduce sums gradients across all ranks
            # We then divide by world_size to get the average gradient
            dist.all_reduce(param.grad.data, op=dist.ReduceOp.SUM)
            param.grad.data /= world_size

    # Optimizer step (updates model parameters identically on all ranks)
    optimizer.step()
    optimizer.zero_grad()

    print(f"Rank {rank}: Local Loss = {loss.item():.4f}, First Layer Grad Norm = {model.layer1.weight.grad.norm().item():.4f}")

    dist.destroy_process_group()


# --- 2. Model Parallelism (Illustrative Concept) --- #

class PipelinedModel(nn.Module):
    def __init__(self, device0, device1):
        super().__init__()
        self.device0 = device0
        self.device1 = device1

        # Part 1 of the model on device0
        self.layer_part1 = nn.Sequential(
            nn.Linear(INPUT_DIM, HIDDEN_DIM),
            nn.ReLU()
        ).to(device0)

        # Part 2 of the model on device1
        self.layer_part2 = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, OUTPUT_DIM)
        ).to(device1)

    def forward(self, x):
        # Input starts on device0
        x = x.to(self.device0)
        
        # Process on device0
        intermediate_output = self.layer_part1(x)
        
        # Move intermediate output to device1
        intermediate_output = intermediate_output.to(self.device1)
        
        # Process on device1
        final_output = self.layer_part2(intermediate_output)
        
        return final_output


def run_model_parallel_concept():
    print("\n--- Model Parallelism (Conceptual Illustration) ---")
    
    # Simulate two devices (e.g., two GPUs)
    # In a real scenario, these would be actual device IDs like 'cuda:0', 'cuda:1'
    # For CPU-only simulation, we just use 'cpu'
    device0 = torch.device("cpu") # or 'cuda:0'
    device1 = torch.device("cpu") # or 'cuda:1'

    # If CUDA is available, use actual GPUs for better illustration
    if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
        device0 = torch.device("cuda:0")
        device1 = torch.device("cuda:1")
        print(f"Using actual GPUs: {device0}, {device1}")
    elif torch.cuda.is_available() and torch.cuda.device_count() == 1:
        print("Warning: Only one CUDA device found. Model parallelism will be simulated on CPU.")
    else:
        print("No CUDA devices found. Model parallelism will be simulated on CPU.")

    model = PipelinedModel(device0, device1)
    
    # Create dummy input data
    dummy_input = torch.randn(BATCH_SIZE_PER_RANK, INPUT_DIM)
    dummy_labels = torch.randint(0, OUTPUT_DIM, (BATCH_SIZE_PER_RANK,))

    # Perform a forward pass
    outputs = model(dummy_input)
    
    # Conceptual backward pass (requires careful handling of gradients across devices)
    # For simplicity, we'll just show the forward pass flow.
    
    print(f"Input on device: {dummy_input.device}")
    print(f"Layer Part 1 on device: {model.layer_part1[0].weight.device}")
    print(f"Layer Part 2 on device: {model.layer_part2[0].weight.device}")
    print(f"Output on device: {outputs.device}")
    print(f"Output shape: {outputs.shape}")

    # Calculate a dummy loss to show it's possible
    criterion = nn.CrossEntropyLoss()
    loss = criterion(outputs.to(device1), dummy_labels.to(device1)) # Ensure labels are on the same device as outputs
    print(f"Conceptual Loss: {loss.item():.4f}")

    # In a real model parallel setup, backward would involve moving gradients back
    # loss.backward() # This would work if all tensors were on the same device or handled by a framework


# --- Main Execution --- #
if __name__ == '__main__':
    print("--- Data Parallelism (Simulated with torch.multiprocessing) ---")
    # For data parallelism, we use multiprocessing to simulate multiple ranks
    # This requires 'nccl' backend for GPU or 'gloo' for CPU
    # Ensure you have enough GPUs if using 'nccl' and WORLD_SIZE > 1
    if torch.cuda.is_available() and torch.cuda.device_count() >= WORLD_SIZE:
        print(f"Running data parallelism on {WORLD_SIZE} CUDA devices.")
        mp.spawn(run_data_parallel_rank, args=(WORLD_SIZE,), nprocs=WORLD_SIZE, join=True)
    else:
        print(f"Warning: Not enough CUDA devices ({torch.cuda.device_count()}) for WORLD_SIZE={WORLD_SIZE}. Running data parallelism on CPU with 'gloo' backend.")
        # Fallback to gloo for CPU simulation if not enough GPUs
        os.environ['MASTER_ADDR'] = 'localhost'
        os.environ['MASTER_PORT'] = '29501' # Use a different port
        dist.init_process_group("gloo", rank=0, world_size=1) # Initialize for rank 0 to avoid error
        dist.destroy_process_group() # Destroy immediately
        
        # Re-spawn with gloo backend
        def run_data_parallel_rank_gloo(rank, world_size):
            os.environ['MASTER_ADDR'] = 'localhost'
            os.environ['MASTER_PORT'] = '29501'
            dist.init_process_group("gloo", rank=rank, world_size=world_size)
            print(f"Rank {rank} (CPU) initialized for data parallelism.")
            
            model = SimpleModel()
            optimizer = optim.SGD(model.parameters(), lr=0.01)
            criterion = nn.CrossEntropyLoss()

            dummy_data = torch.randn(GLOBAL_BATCH_SIZE, INPUT_DIM)
            dummy_labels = torch.randint(0, OUTPUT_DIM, (GLOBAL_BATCH_SIZE,))

            start_idx = rank * BATCH_SIZE_PER_RANK
            end_idx = (rank + 1) * BATCH_SIZE_PER_RANK
            local_data = dummy_data[start_idx:end_idx]
            local_labels = dummy_labels[start_idx:end_idx]

            outputs = model(local_data)
            loss = criterion(outputs, local_labels)
            loss.backward()

            for param in model.parameters():
                if param.grad is not None:
                    dist.all_reduce(param.grad.data, op=dist.ReduceOp.SUM)
                    param.grad.data /= world_size

            optimizer.step()
            optimizer.zero_grad()
            print(f"Rank {rank} (CPU): Local Loss = {loss.item():.4f}, First Layer Grad Norm = {model.layer1.weight.grad.norm().item():.4f}")
            dist.destroy_process_group()

        mp.spawn(run_data_parallel_rank_gloo, args=(WORLD_SIZE,), nprocs=WORLD_SIZE, join=True)

    run_model_parallel_concept()


### Interpreting the Code and Performance Trade-offs

#### Data Parallelism Code Interpretation

The provided code simulates **data parallelism** using `torch.multiprocessing.spawn` to create multiple processes, each acting as a distinct 'rank' (representing a GPU or node). 

1.  **`dist.init_process_group`**: Each process initializes a distributed environment, allowing them to communicate.
2.  **`SimpleModel().to(rank)`**: Each rank instantiates its *own copy* of the `SimpleModel` and moves it to its assigned device (e.g., `cuda:0`, `cuda:1`, or `cpu`).
3.  **Data Sharding**: The `dummy_data` and `dummy_labels` are conceptually sharded. Each rank receives a unique `local_data` and `local_labels` slice from the global batch.
4.  **Local Computation**: Each rank performs a forward and backward pass independently on its local data, computing `loss` and `param.grad` for its local model replica.
5.  **`dist.all_reduce(param.grad.data, op=dist.ReduceOp.SUM)`**: This is the core of data parallelism. After local gradient computation, all ranks collectively sum their gradients for each parameter. The result is then divided by `world_size` to get the average gradient. This ensures that all model replicas receive the same, synchronized gradient update.
6.  **`optimizer.step()`**: Each rank updates its local model parameters using the averaged gradients. Because all ranks apply the same average gradient, their model parameters remain synchronized.

**Output**: You'll see output from each rank, showing its local loss and the norm of the first layer's gradients. Notice how the gradient norms, after `all_reduce`, would be identical across all ranks, demonstrating synchronization.

#### Model Parallelism Code Interpretation

The **model parallelism** section is a conceptual illustration rather than a fully runnable distributed example due to the complexity of setting up true distributed model parallelism in a single, self-contained cell. 

1.  **`PipelinedModel`**: This model is designed to have two distinct parts (`layer_part1`, `layer_part2`).
2.  **Device Assignment**: `layer_part1` is explicitly moved to `device0` and `layer_part2` to `device1`. This simulates partitioning the model across different devices.
3.  **Data Movement**: In the `forward` method, the `intermediate_output` from `layer_part1` is explicitly moved from `device0` to `device1` before being processed by `layer_part2`. This highlights the crucial communication of activations between devices in model parallelism.

**Output**: The output shows which device each part of the model resides on and the device of the final output. This demonstrates the conceptual flow of data through a partitioned model.

#### Performance Trade-offs

| Feature             | Data Parallelism (e.g., DDP, FSDP)                               | Model Parallelism (e.g., Pipeline, Tensor)                               |
| :------------------ | :--------------------------------------------------------------- | :----------------------------------------------------------------------- |
| **Primary Goal**    | Accelerate training throughput (larger effective batch size)     | Train models too large for a single device's memory                      |
| **Memory Usage**    | Each device stores full model parameters, gradients, optimizer states (unless FSDP) | Each device stores only a *part* of the model                            |
| **Communication**   | Gradients (all-reduce)                                           | Intermediate activations (forward) and gradients (backward) between layers/parts |
| **Overhead**        | Communication latency for gradient synchronization               | Communication latency for activations, pipeline bubbles (idle time)      |
| **Scalability**     | Good for scaling computation, limited by single-device model size | Good for scaling model size, can be complex to balance workload          |
| **Ease of Use**     | Generally simpler to implement (e.g., `DDP` wrapper)             | More complex, requires careful model partitioning and scheduling         |

#### Typical Use Cases

*   **Data Parallelism**: Ideal when the model *can* fit on a single GPU, but you want to train faster by using more data per step or by reducing wall-clock time. FSDP extends this to models that are slightly too large for a single GPU by sharding parameters.
*   **Model Parallelism**: Essential when the model's parameters or activations are too large to fit into the memory of a single GPU. This is increasingly common for LLMs with hundreds of billions or trillions of parameters. It's often combined with data parallelism for optimal performance.

Modern LLM training often employs **hybrid parallelism**, combining FSDP (for data and parameter sharding) with pipeline parallelism (for splitting layers across nodes) and sometimes tensor parallelism (for splitting large layers within a node). Frameworks like DeepSpeed and PyTorch's distributed modules abstract much of this complexity, allowing researchers and engineers to focus on model architecture and data.


### Resources

*   **PyTorch Distributed Documentation**: The official guide for `torch.distributed` and related modules.
    *   [PyTorch Distributed Overview](https://pytorch.org/docs/stable/distributed.html)
    *   [PyTorch DDP Tutorial](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html)
    *   [PyTorch FSDP Tutorial](https://pytorch.org/tutorials/intermediate/FSDP_tutorial.html)
    *   [PyTorch Tensor Parallelism](https://pytorch.org/docs/stable/tensor_parallel.html)
    *   [PyTorch Pipeline Parallelism](https://pytorch.org/docs/stable/pipeline.html)

*   **Hugging Face Accelerate**: A library that simplifies distributed training for PyTorch models, abstracting away much of the boilerplate.
    *   [Hugging Face Accelerate Documentation](https://huggingface.co/docs/accelerate/index)

*   **DeepSpeed**: Microsoft's deep learning optimization library, offering advanced distributed training techniques (ZeRO, pipeline parallelism, etc.).
    *   [DeepSpeed GitHub](https://github.com/microsoft/DeepSpeed)
    *   [DeepSpeed Documentation](https://www.deepspeed.ai/)

*   **Megatron-LM**: NVIDIA's framework for training large transformer models, pioneering many model parallelism techniques.
    *   [Megatron-LM GitHub](https://github.com/NVIDIA/Megatron-LM)

*   **Google AI Studio / JAX Distributed**: While the code examples are PyTorch-centric, understanding distributed training in JAX is also crucial for 2026.
    *   [JAX Distributed Overview](https://jax.readthedocs.io/en/latest/jax-101/06-parallelism.html)
    *   [Google AI Blog on LLM Training](https://ai.googleblog.com/search/label/Large%20Language%20Models) (Search for distributed training related posts)
